<a href="https://colab.research.google.com/github/MaineCalabrezi13/DiagnosticoPlantas/blob/main/diagnostico_de_plantas.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install streamlit
!pip install pillow
!pip install pyngrok
!pip install ultralytics
!pip install kaggle

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 57.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 102.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.3/41.3 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 21.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.2/53.2 kB 4.1 MB/s eta 0:00:00


In [ ]:
%%writefile app.py

import streamlit as st
from ultralytics import YOLO
from PIL import Image
import os

# ============================================
# CONFIGURAÇÕES DA PÁGINA
# ============================================
st.set_page_config(
    page_title="AgroScan AI",
    page_icon="🌱",
    layout="centered"
)

# ============================================
# CARREGAR MODELOS TREINADOS (EM CASCATA)
# ============================================
@st.cache_resource
def carregar_modelos():
    # 1. Carrega o modelo binário (Treinado com as classes: 'leaf' e 'non-leaf')
    modelo_validador = YOLO("best_nao_folha.pt")

    # 2. Carrega o modelo de doenças (Pimentão, Batata e Tomate)
    modelo_diagnostico = YOLO("best.pt")

    return modelo_validador, modelo_diagnostico

try:
    modelo_validador, modelo_diagnostico = carregar_modelos()
except Exception as erro:
    st.error(f"Erro ao carregar os modelos: {erro}")
    st.stop()

# ============================================
# DICIONÁRIO DE EXPLICAÇÕES (DIAGNÓSTICOS)
# ============================================
RECOMENDACOES = {
    # --- PIMENTÃO ---
    "Pepper__bell___Bacterial_spot": {
        "diagnostico": "Pimentão — Mancha bacteriana",
        "explicacao": "Foram identificadas lesões foliares úmidas e escuras características de infecção bacteriana.",
        "acao": "Eliminar restos culturais contaminados, evitar irrigação por aspersão excessiva e utilizar produtos à base de cobre se recomendado."
    },
    "Pepper__bell___healthy": {
        "diagnostico": "Pimentão — Saudável",
        "explicacao": "A folha do pimentão apresenta coloração e aspectos saudáveis, sem sinais detectáveis de fitopatologias.",
        "acao": "Manter o manejo nutricional e monitoramento periódico preventivo."
    },
    # --- BATATA ---
    "Potato___Early_blight": {
        "diagnostico": "Batata — Pinta preta / Alternariose",
        "explicacao": "Foram detectadas manchas circulares de cor marrom ou preta com anéis concêntricos nas folhas mais velhas.",
        "acao": "Aplicar fungicidas protetores ou sistêmicos específicos, evitar o plantio adensado e monitorar a umidade foliar."
    },
    "Potato___Late_blight": {
        "diagnostico": "Batata — Requeima",
        "explicacao": "Identificados sintomas severos de queima foliar com áreas necróticas úmidas, indicando alta agressividade do patógeno.",
        "acao": "Realizar controle químico fitossanitário imediato com fungicidas apropriados e remover focos iniciais da doença."
    },
    "Potato___healthy": {
        "diagnostico": "Batata — Saudável",
        "explicacao": "A folhagem da batateira apresenta-se em ótimas condições de vigor, sem sintomas aparentes.",
        "acao": "Continuar o monitoramento sistemático e as boas práticas de manejo fitotécnico."
    },
    # --- TOMATE ---
    "Tomato_Bacterial_spot": {
        "diagnostico": "Tomate — Mancha bacteriana",
        "explicacao": "Pequenas manchas escuras e angulares circundadas por halos amarelados foram encontradas na folha.",
        "acao": "Evitar o trabalho nos campos com plantas molhadas e pulverizar defensivos recomendados à base de cobre."
    },
    "Tomato_Early_blight": {
        "diagnostico": "Tomate — Pinta preta / Alternariose",
        "explicacao": "Constatada a presença de lesões foliares com pontuações necróticas concêntricas típicas de alternariose.",
        "acao": "Promover a rotação de culturas, podar as folhas baixeiras afetadas e aplicar fungicidas adequados."
    },
    "Tomato_Late_blight": {
        "diagnostico": "Tomate — Requeima",
        "explicacao": "Foram identificadas manchas grandes e irregulares de aspecto esverdeado-escuro ou oleoso nas folhas.",
        "acao": "Pulverizar fungicidas sistêmicos preventivos/curativos e otimizar o arejamento da cultura."
    },
    "Tomato_Leaf_Mold": {
        "diagnostico": "Tomate — Bolor cinzento / Mofo das folhas",
        "explicacao": "Presença de uma cobertura aveludada cinza a esverdeada na face inferior da folha, associada a manchas amareladas na face superior.",
        "acao": "Reduzir a umidade relativa dentro da estufa/lavoura e melhorar o espaçamento entre plantas."
    },
    "Tomato_Septoria_leaf_spot": {
        "diagnostico": "Tomate — Septoriose",
        "explicacao": "Foram detectadas numerosas manchas pequenas, circulares, com centros cinzentos e bordas escuras.",
        "acao": "Eliminar restos de plantas após a colheita e aplicar o controle químico apropriado."
    },
    "Tomato_Spider_mites_Two_spotted_spider_mite": {
        "diagnostico": "Tomate — Ácaro-rajado",
        "explicacao": "Sinais de bronzeamento, pequenas pontuações claras nas folhas e presença de teias finas causadas por ácaros.",
        "acao": "Aplicar acaricidas específicos de forma localizada ou introduzir predadores naturais."
    },
    "Tomato__Target_Spot": {
        "diagnostico": "Tomate — Mancha-alvo",
        "explicacao": "Identificadas lesões circulares planas com círculos concêntricos bem definidos.",
        "acao": "Garantir uma boa drenagem do solo e aplicar fungicidas."
    },
    "Tomato__Tomato_YellowLeaf__Curl_Virus": {
        "diagnostico": "Tomate — Vírus do enrolamento amarelo",
        "explicacao": "Folhas superiores exibem redução de tamanho e enrolamento.",
        "acao": "Controlar a mosca-branca."
    },
    "Tomato__Tomato_mosaic_virus": {
        "diagnostico": "Tomate — Vírus do mosaico",
        "explicacao": "Folhas apresentam padrão de mosaico.",
        "acao": "Remover plantas contaminadas."
    },
    "Tomato_healthy": {
        "diagnostico": "Tomate — Saudável",
        "explicacao": "Folha sem anomalias aparentes.",
        "acao": "Manter o plano atual."
    }
}

# ============================================
# TÍTULO DA INTERFACE
# ============================================
st.title("🌱 AgroScan AI")
st.subheader("Diagnóstico Automatizado de Fitopatologias")
st.write("Faça o upload da foto de uma folha (Pimentão, Batata ou Tomate) para análise instantânea.")
st.divider()

# ============================================
# UPLOAD DA IMAGEM
# ============================================
imagem = st.file_uploader("Selecione uma imagem", type=["jpg", "jpeg", "png", "jfif"])

if imagem:
    # 1. Carrega a imagem original
    img = Image.open(imagem)

    # 2. Cria uma cópia padronizada apenas para exibição na tela
    img_exibicao = img.copy()
    TAMANHO_MAXIMO = (400, 400) # Define a largura e altura máxima padrão em pixels
    img_exibicao.thumbnail(TAMANHO_MAXIMO)

    # Criando as colunas para o layout
    col1, col2 = st.columns(2)

    with col1:
        st.subheader("Imagem enviada")

        # 3. Exibe a imagem padronizada.
        # Mudamos use_container_width para False para respeitar o tamanho do thumbnail
        st.image(img_exibicao, use_container_width=False)

        st.write(f"📄 Nome: {imagem.name}")
        # Mostra a resolução real da foto original enviada
        st.write(f"📐 Resolução Original: {img.width} x {img.height}")

    # ============================================
    # PROCESSAMENTO DE INTELIGÊNCIA ARTIFICIAL
    # ============================================
    with st.spinner("Analisando imagem..."):
        # Converter e salvar imagem temporária para o YOLO ler
        caminho_temp = "imagem_temp.jpg"
        if img.mode in ("RGBA", "P"):
            img_para_salvar = img.convert("RGB")
        else:
            img_para_salvar = img
        img_para_salvar.save(caminho_temp, "JPEG")

        # --- PASSO 1: VERIFICAÇÃO SE É FOLHA ---
        res_validador = modelo_validador(caminho_temp, verbose=False)[0]
        id_classe_val = res_validador.probs.top1
        classe_validador = res_validador.names[id_classe_val].lower() # Pega o nome ('leaf' ou 'non-leaf')
        conf_validador = float(res_validador.probs.top1conf) * 100

        # Validação Exata baseada no seu dataset binário
        eh_folha = (classe_validador == "leaf")

        nome_classe = None
        confianca = 0.0

        # --- PASSO 2: DIAGNÓSTICO (SÓ SE FOR FOLHA CONFIRMADA) ---
        if eh_folha:
            res_diagnostico = modelo_diagnostico(caminho_temp, verbose=False)[0]
            id_classe_diag = res_diagnostico.probs.top1
            nome_classe = res_diagnostico.names[id_classe_diag]
            confianca = float(res_diagnostico.probs.top1conf) * 100

        # Remove arquivo temporário
        if os.path.exists(caminho_temp):
            os.remove(caminho_temp)

   # ============================================
    # EXIBIÇÃO DOS RESULTADOS (OTIMIZADA E COMPACTA)
    # ============================================
    with col2:
        st.subheader("Resultado do AgroScan")

        if not eh_folha:
            # Título do erro menor e direto ao ponto
            st.error("#### ❌ Análise Interrompida")
            st.markdown("O objeto ou imagem enviada **não** foi reconhecida como uma folha válida.")

            # Card de Destaque Otimizado (Compacto)
            with st.container(border=True):
                st.markdown(
                    """
                    <div style='line-height: 1.2;'>
                        <small style='color: #808495;'>🌐 FILTRO DE REJEIÇÃO</small>
                        <h4 style='margin: 4px 0 12px 0; color: #31333F;'>Não-Folha (non-leaf)</h4>
                        <hr style='margin: 8px 0; border: none; border-top: 1px solid #e6eaf1;'>
                        <small style='color: #808495;'>Confiança do Bloqueio:</small>
                    </div>
                    """,
                    unsafe_allow_html=True
                )
                st.progress(int(conf_validador))
                st.caption(f"Grau de certeza: **{conf_validador:.2f}%**")

        else:
            # Layout de Sucesso Compacto
            st.success("✨ **Folha Validada com Sucesso!**")

            if confianca < 50:
                st.warning("#### ⚠️ Confiança Insuficiente")
                st.markdown("A IA identificou a folha, mas a precisão do diagnóstico está abaixo de **50%**.")

                with st.container(border=True):
                    st.markdown(
                        f"""
                        <div style='line-height: 1.2;'>
                            <small style='color: #808495;'>DIAGNÓSTICO INCONCLUSIVO</small>
                            <h4 style='margin: 4px 0 12px 0; color: #31333F;'>Confiança Baixa ({confianca:.2f}%)</h4>
                        </div>
                        """,
                        unsafe_allow_html=True
                    )
                    st.progress(int(confianca))
                    st.caption("⚠️ Mínimo aceitável para o manejo: 50.00%")

                st.info("💡 **Dica:** Aproxime a câmera da folha afetada e garanta boa iluminação para um novo teste.")

            else:
                if nome_classe in RECOMENDACOES:
                    info = RECOMENDACOES[nome_classe]
                    eh_saudavel = "healthy" in nome_classe
                    icon = "🟢" if eh_saudavel else "🔴"

                    # Card de Diagnóstico Principal Compacto
                    with st.container(border=True):
                        st.markdown(
                            f"""
                            <div style='line-height: 1.2;'>
                                <small style='color: #808495;'>📋 STATUS DO DIAGNÓSTICO</small>
                                <h4 style='margin: 4px 0 12px 0; color: #31333F;'>{icon} {info['diagnostico']}</h4>
                                <hr style='margin: 8px 0; border: none; border-top: 1px solid #e6eaf1;'>
                                <small style='color: #808495;'>Certeza do Modelo:</small>
                            </div>
                            """,
                            unsafe_allow_html=True
                        )
                        st.progress(int(confianca))
                        st.caption(f"Confiança: **{confianca:.2f}%**")

                    # Seções textuais limpas e bem espaçadas
                    st.markdown("<p style='margin-bottom: -5px;'><strong>Análise Visual</strong></p>", unsafe_allow_html=True)
                    st.write(info["explicacao"])

                    st.markdown("<p style='margin-bottom: -5px;'><strong>Plano de Ação Recomendado</strong></p>", unsafe_allow_html=True)
                    if eh_saudavel:
                        st.info(info["acao"])
                    else:
                        st.warning(info["acao"])

                else:
                    st.error(f"Classe reconhecida (`{nome_classe}`), mas sem mapeamento no dicionário.")
st.divider()
st.caption("AgroScan AI © 2026")

Overwriting app.py


In [ ]:
!streamlit run app.py &>/content/logs.txt &

In [ ]:
from pyngrok import ngrok

ngrok.set_auth_token(
"3Eu0uGhV8Sw9rvvCGXlcwhNKIfj_81kRpesKtmutVx3YxxhhF"
)

In [ ]:
from pyngrok import ngrok

public_url = ngrok.connect(8501)

print(public_url)

NgrokTunnel: "https://slideshow-litter-outspoken.ngrok-free.dev" -> "http://localhost:8501"
